# SAM3 isolated test

Standalone smoke test for **SAM3 text-prompted (concept) segmentation**. This does *not* import the obstacle pipeline -- it exercises SAM3 directly so we can confirm the model loads and returns masks for a text query.

Two backends are shown:
1. **Ultralytics** `SAM3SemanticPredictor` -- matches what `obstacle_detection/handlers/sam3_handler.py` uses; weights auto-resolve from `sam3.pt`.
2. **Official Meta `sam3`** package (from https://github.com/facebookresearch/sam3) -- as an alternative.

Paths refer to the **Azure server** filesystem.

## 1. Environment check

In [ ]:
import torch, ultralytics

print("torch      :", torch.__version__)
print("ultralytics:", ultralytics.__version__)
print("cuda avail :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device     :", torch.cuda.get_device_name(0))

## 2. Config

Point at one test image and a few text concepts to query.

In [ ]:
import os

BASE_PATH       = "/mnt/aitraining/krishna/2026/obstacle_pipeline"  # project root -> models land here
TEST_IMAGES_DIR = os.path.join(BASE_PATH, "test_images")
SAM3_WEIGHTS    = os.path.join(BASE_PATH, "sam3.pt")

# --- force model downloads into the project root ---
# ultralytics auto-downloads bare weight names into the CWD, so cd there:
os.chdir(BASE_PATH)
# Meta sam3 pulls from the HuggingFace hub -> cache it under the project root too:
os.environ["HF_HOME"] = os.path.join(BASE_PATH, "hf_cache")

IMAGE_NAME = "0e63ccd4-ea08-49b4-8578-ee013f9a31b7.png"
IMAGE_PATH = os.path.join(TEST_IMAGES_DIR, IMAGE_NAME)
TEXTS      = ["person", "car", "bus"]

# If the weights already exist use the full path; otherwise ultralytics
# downloads "sam3.pt" into the CWD we just set to BASE_PATH.
MODEL = SAM3_WEIGHTS if os.path.exists(SAM3_WEIGHTS) else "sam3.pt"
assert os.path.exists(IMAGE_PATH), f"image not found: {IMAGE_PATH}"
print("cwd   :", os.getcwd())
print("image :", IMAGE_PATH)
print("model :", MODEL)
print("texts :", TEXTS)

## 3. Run SAM3 (ultralytics backend)

`set_image` once, then query the loaded image with the text prompts.

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor

overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model=MODEL,
    half=True,      # FP16 for faster inference
    save=False,
)
predictor = SAM3SemanticPredictor(overrides=overrides)
predictor.set_image(IMAGE_PATH)   # set once, query many
results = predictor(text=TEXTS)
results

## 4. Inspect results

In [ ]:
r = results[0]
n = 0 if r.masks is None else len(r.masks)
print(f"detections: {n}")
if n:
    print("masks shape:", tuple(r.masks.data.shape))   # (N, H, W)
    print("boxes xyxy :", r.boxes.xyxy.cpu().numpy().round(1).tolist())
    print("conf       :", r.boxes.conf.cpu().numpy().round(3).tolist())
    print("class ids  :", r.boxes.cls.cpu().numpy().astype(int).tolist())
    print("names      :", r.names)

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt

# r.plot() returns a BGR annotated image -> flip to RGB for matplotlib
annotated = r.plot()[:, :, ::-1]
plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"SAM3: {TEXTS}")
plt.show()